# PhysGraph Phase 1 — Kaggle / Colab runner

This notebook runs the **exact same FEM code** as your Arch Linux repository. Phase 1 intentionally uses CPU; later neural training will use the cloud GPU.

**Before starting:**
- **Colab:** upload `physgraph-phase01.zip` (Files sidebar) to `/content/`.
- **Kaggle:** add the ZIP as a notebook Dataset input (for example `/kaggle/input/physgraph-starter/physgraph-phase01.zip`).
- Or provide a public Git repository address in `REPO_URL` below (requires internet-enabled runtime).

The ZIP contains a top-level `physgraph/` folder. Run each cell in order and save/download `outputs/` before your session expires.

The current checkout also runs the Phase 2A elliptic CPU reference. Use a current archive or Git checkout containing scripts/run_phase2.py.


In [ ]:
from pathlib import Path
import os, subprocess, sys, zipfile

BASE = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content') if Path('/content').exists() else Path.cwd()
ARCHIVE_PATH = ''  # Optional: explicit full path to physgraph-phase01.zip
REPO_URL = ''      # Alternative: your public GitHub repository URL

archive = Path(ARCHIVE_PATH) if ARCHIVE_PATH else None
if archive is None:
    candidates = [BASE / 'physgraph-phase01.zip', Path('/content/physgraph-phase01.zip')]
    kaggle_inputs = Path('/kaggle/input')
    if kaggle_inputs.exists():
        candidates += list(kaggle_inputs.glob('**/physgraph-phase01.zip'))
    archive = next((p for p in candidates if p.exists()), None)

if archive is not None and archive.exists():
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(BASE)
    ROOT = BASE / 'physgraph'
elif REPO_URL:
    ROOT = BASE / 'physgraph'
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(ROOT)], check=True)
else:
    raise FileNotFoundError('Upload physgraph-phase01.zip, set ARCHIVE_PATH, or set REPO_URL')

assert (ROOT / 'pyproject.toml').exists(), f'Not a PhysGraph root: {ROOT}'
print('PhysGraph root:', ROOT)


## Install package and validate numerical tests

The standard Kaggle/Colab images usually contain NumPy, SciPy, Matplotlib and pytest. We install the local package without dependency downloads first. If Python reports missing libraries, run `python -m pip install -e '.[dev]'` with notebook internet enabled.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-build-isolation', '--no-deps', '-e', str(ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=ROOT, check=True)


## Run FEM baseline, exact comparison, convergence checks

These are CPU-only experiments. Later phases will introduce PyTorch and explicit device selection.


In [ ]:
subprocess.run([sys.executable, 'scripts/cloud_smoke.py'], cwd=ROOT, check=True)
subprocess.run([sys.executable, 'scripts/run_phase1.py', '--config', 'configs/experiments/poisson_cloud.json'], cwd=ROOT, check=True)
subprocess.run([sys.executable, 'scripts/run_phase2.py', '--config', 'configs/experiments/elliptic_cloud.json'], cwd=ROOT, check=True)


## Inspect the generated FEM comparison image

In [ ]:
from IPython.display import display, Image
display(Image(filename=str(ROOT / 'outputs/phase1_cloud/comparison.png')))


## Archive results before the cloud session ends

In [ ]:
import shutil
archive_out = shutil.make_archive(str(BASE / 'physgraph-phase1-results'), 'zip', root_dir=ROOT, base_dir='outputs')
print('Saved results archive:', archive_out)
print('Kaggle: save notebook output or download the ZIP. Colab: use the Files sidebar or files.download(archive_out).')
